# AI Word Correction — Visual Step-by-Step
Each cell is one step of the pipeline. Run them top-to-bottom.

In [1]:
# ── STEP 0: CONFIG ─────────────────────────────────────────────────────────
import sys, os, io, json, base64, pathlib

# Add project root to path so word_constructor is importable
PROJECT_ROOT = str(pathlib.Path("__file__").resolve().parent.parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Load .env (picks up OPENAI_API_KEY, model, base_url, etc.)
env_path = pathlib.Path(PROJECT_ROOT) / ".env"
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, _, v = line.partition("=")
            os.environ.setdefault(k.strip(), v.strip())
    print(f"Loaded .env from {env_path}")
else:
    print(f"No .env at {env_path} — set OPENAI_API_KEY manually if needed")

# ── Load from example_replacement.json ────────────────────────────────────
EXAMPLE_JSON = pathlib.Path(PROJECT_ROOT) / "example_replacement.json"
with EXAMPLE_JSON.open(encoding="utf-8") as f:
    example = json.load(f)

# Decode embedded docx bytes
DOC_BYTES = io.BytesIO(base64.b64decode(example["content_base64"]))
DOC_FILENAME = example.get("filename", "template.docx")

# Placeholders: keep only string values (skip МассивШапки lists, meta keys)
_META_KEYS = {"ИспользоватьAI", "PromtAI"}
PARAMS: dict[str, str] = {
    k: str(v)
    for k, v in example["placeholders"].items()
    if isinstance(v, str) and k not in _META_KEYS
}

PROMPT_AI: str = example.get("PromtAI", "")
USE_AI: bool = str(example.get("UseAI", "true")).lower() == "true"

print(f"\nFile    : {DOC_FILENAME}")
print(f"UseAI   : {USE_AI}")
print(f"PromtAI : {PROMPT_AI[:80]!r}{'…' if len(PROMPT_AI) > 80 else ''}")
print(f"\nParams ({len(PARAMS)} placeholders):")
for k, v in list(PARAMS.items())[:15]:
    print(f"  [{k}] = {str(v)[:70]!r}")
if len(PARAMS) > 15:
    print(f"  … and {len(PARAMS) - 15} more")

Loaded .env from C:\Users\User\Desktop\word-portal\.env

File    : template.docx
UseAI   : True
PromtAI : 'Дополнительные правила для этого документа (приоритет выше общих правил, если ес'…

Params (9 placeholders):
  [Ссылка] = 'Прием на работу 0000-000233 от 15.12.2025'
  [СсылкаДолжностьРуководителяНаименование] = 'hr бизнес-партнер'
  [СсылкаДатаПриема] = '15.12.2025'
  [РеквизитыСотрудникПодразделениеНаименование] = 'компас воад'
  [СсылкаСотрудникФизическоеЛицоФИО] = 'Садық Ермек Жәнібекұлы'
  [РеквизитыОтветственныйФизическоеЛицоФИО] = 'Сатымбаева Салтанат Мусаевна'
  [РеквизитыРуководительФИО] = 'Есжанова Зарина Серикалиевна'
  [РеквизитыДолжностьРуководителяНаименование] = 'hr бизнес-партнер'
  [РеквизитыСотрудникДолжностьНаименование] = 'кассир-повар'


In [2]:
# ── STEP 1: LOAD DOCUMENT ──────────────────────────────────────────────────
from docx import Document

doc = Document(DOC_BYTES)

para_count = len(doc.paragraphs)
table_count = len(doc.tables)
section_count = len(doc.sections)

print(f"Document loaded: {DOC_FILENAME}")
print(f"  Paragraphs : {para_count}")
print(f"  Tables     : {table_count}")
print(f"  Sections   : {section_count}")

print("\nFirst 5 non-empty paragraphs:")
shown = 0
for p in doc.paragraphs:
    text = p.text.strip()
    if text:
        print(f"  {text[:120]}")
        shown += 1
        if shown >= 5:
            break

Document loaded: template.docx
  Paragraphs : 32
  Tables     : 2
  Sections   : 1

First 5 non-empty paragraphs:
  О приеме на работу
  В соответствии со статьей 34 Трудового кодекса Республики Казахстан, ПРИКАЗЫВАЮ:
  Принять [СсылкаСотрудникФизическоеЛицоФИО] на [РеквизитыСотрудникДолжностьНаименование] [РеквизитыСотрудникПодразделение
  Основание: трудовой договор № [НомерДоговора] от [ДатаНачалаДоговора]года, заявление от [СсылкаСотрудникФизическоеЛицоФИ
  С приказом ознакомлен (а):


In [3]:
# ── STEP 2: WALK DOCUMENT → TextUnits ─────────────────────────────────────
from word_constructor.ai_correction import walk_document

text_units = walk_document(doc)

print(f"Total TextUnits: {len(text_units)}")
print()

type_counts: dict[str, int] = {}
for u in text_units:
    type_counts[u.source_type] = type_counts.get(u.source_type, 0) + 1
print("By source_type:")
for t, c in type_counts.items():
    print(f"  {t:25s} {c}")

print()
print("── First 10 non-empty units ──")
shown = 0
for u in text_units:
    if u.text.strip():
        ai_flag = " [AI-EXCLUDED]" if u.ai_excluded else ""
        print(f"  [{u.source_type}] {u.source_path}{ai_flag}")
        print(f"    {u.text[:100]!r}")
        shown += 1
        if shown >= 10:
            break

Total TextUnits: 46

By source_type:
  body_paragraph            32
  table_cell                14

── First 10 non-empty units ──
  [table_cell] table[0].row[0].cell[0]
    'БҰЙРЫҚ'
  [table_cell] table[0].row[0].cell[3]
    'ПРИКАЗ'
  [table_cell] table[0].row[1].cell[0]
    '<ДатаДокумента> года'
  [table_cell] table[0].row[1].cell[1]
    '<ДатаДокумента> года'
  [table_cell] table[0].row[1].cell[2]
    '№ <РегНомерДокумента>'
  [table_cell] table[0].row[1].cell[3]
    '№ <РегНомерДокумента>'
  [table_cell] table[0].row[2].cell[0]
    'Алматы қаласы'
  [table_cell] table[0].row[2].cell[1]
    'Алматы қаласы'
  [table_cell] table[0].row[2].cell[2]
    'город Алматы'
  [table_cell] table[0].row[2].cell[3]
    'город Алматы'


In [4]:
# ── STEP 3: FIND OCCURRENCES ──────────────────────────────────────────────
from word_constructor.ai_correction import find_occurrences

occurrences = find_occurrences(text_units, PARAMS)

print(f"Found {len(occurrences)} occurrence(s)")
print()

for occ in occurrences:
    excluded = " ⛔ AI-EXCLUDED" if occ.ai_excluded else ""
    print(f"  id             : {occ.id}")
    print(f"  placeholder    : [{occ.placeholder}]")
    print(f"  occurrence_idx : {occ.occurrence_index}")
    print(f"  original_value : {occ.original_value!r}")
    print(f"  source_type    : {occ.source_type}")
    print(f"  source_path    : {occ.source_path}{excluded}")
    print(f"  context_text   : {occ.context_text[:120]!r}")
    print()

Found 7 occurrence(s)

  id             : СсылкаСотрудникФизическоеЛицоФИО#0
  placeholder    : [СсылкаСотрудникФизическоеЛицоФИО]
  occurrence_idx : 0
  original_value : 'Садық Ермек Жәнібекұлы'
  source_type    : body_paragraph
  source_path    : paragraph[9]
  context_text   : 'Принять [СсылкаСотрудникФизическоеЛицоФИО] на [РеквизитыСотрудникДолжностьНаименование] [РеквизитыСотрудникПодразделение'

  id             : РеквизитыСотрудникДолжностьНаименование#0
  placeholder    : [РеквизитыСотрудникДолжностьНаименование]
  occurrence_idx : 0
  original_value : 'кассир-повар'
  source_type    : body_paragraph
  source_path    : paragraph[9]
  context_text   : 'Принять [СсылкаСотрудникФизическоеЛицоФИО] на [РеквизитыСотрудникДолжностьНаименование] [РеквизитыСотрудникПодразделение'

  id             : РеквизитыСотрудникПодразделениеНаименование#0
  placeholder    : [РеквизитыСотрудникПодразделениеНаименование]
  occurrence_idx : 0
  original_value : 'компас воад'
  source_type    : body_p

In [5]:
# ── STEP 3b: REDUNDANCY / DUPLICATE DETECTION ─────────────────────────────
# When [Должность] already contains the full [Подразделение] value,
# the Подразделение occurrence is marked redundant → will be blanked out.
#
# Detection runs automatically inside find_occurrences / extract_placeholder_occurrences.
# This cell shows what was detected, plus a concrete demo with overlapping values.

print("── Redundancy detection on actual PARAMS ──")
redundant = [o for o in occurrences if o.redundant_in]
if redundant:
    for o in redundant:
        adj_id = o.redundant_in
        adj = next((x for x in occurrences if x.id == adj_id), None)
        print(f"  ⚠  [{o.placeholder}#{o.occurrence_index}]")
        print(f"       value         : {o.original_value!r}")
        print(f"       embedded in   : [{adj.placeholder}#{adj.occurrence_index}] = {adj.original_value!r}" if adj else f"       embedded in   : {adj_id}")
else:
    print("  (no redundancies in current PARAMS — values don't overlap)")

print()
print("── Demo: overlapping values that WOULD trigger dedup ──")
from word_constructor.ai_correction import find_occurrences as _find

# Simulate a document paragraph: "[Должность] [Подразделение] АО «Halyk Finance»"
from word_constructor.ai_correction.types import TextUnit as _TU
demo_units = [
    _TU(
        source_type="body_paragraph",
        source_path="paragraph[demo]",
        text="Принять сотрудника на [ДолжностьДемо] [ПодразделениеДемо] АО «Halyk Finance»",
    )
]
demo_params = {
    "ДолжностьДемо":    "Главный менеджер департамента кадровой политики",
    "ПодразделениеДемо": "департамент кадровой политики",
}
demo_occs = _find(demo_units, demo_params)

for o in demo_occs:
    flag = f"  ⚠  REDUNDANT (embedded in [{o.redundant_in}])" if o.redundant_in else "  ✅ kept"
    print(f"\n  [{o.placeholder}#{o.occurrence_index}]{flag}")
    print(f"    value : {o.original_value!r}")

── Redundancy detection on actual PARAMS ──
  (no redundancies in current PARAMS — values don't overlap)

── Demo: overlapping values that WOULD trigger dedup ──

  [ДолжностьДемо#0]  ✅ kept
    value : 'Главный менеджер департамента кадровой политики'

  [ПодразделениеДемо#0]  ⚠  REDUNDANT (embedded in [ДолжностьДемо#0])
    value : 'департамент кадровой политики'


In [6]:
# ── STEP 4: SANITY CHECK ──────────────────────────────────────────────────
from word_constructor.ai_correction import sanity_check

ok, messages = sanity_check(text_units, occurrences)

status = "✅ PASSED" if ok else "❌ FAILED"
print(f"Sanity check: {status}")
if messages:
    for msg in messages:
        print(f"  {msg}")
else:
    print(f"  raw regex matches == occurrences ({len(occurrences)}) — counts match")

Sanity check: ❌ FAILED
  raw placeholder regex matches=9, occurrences=7


In [7]:
# ── STEP 5: EXTRACT FULL DOCUMENT TEXT (headers + body + footers) ──────────
# This single block is now the entire context sent to AI.
# AI reads the whole document and determines the correct grammatical form
# of each placeholder from the surrounding prose — no per-placeholder snippets needed.

from word_constructor.ai_correction import document_full_text

full_doc_text = document_full_text(doc)

char_count = len(full_doc_text)
line_count = full_doc_text.count("\n") + 1
section_labels = [ln for ln in full_doc_text.splitlines() if ln.startswith("[")]

print(f"Full document text: {char_count} chars, {line_count} lines")
print(f"Sections / text-units included: {len(section_labels)}")
print()
print("── Section labels (headers first, then body) ──")
for label in section_labels:
    print(f"  {label}")
print()
print("── Full text preview (first 1 500 chars) ──")
print(full_doc_text[:1500])

Full document text: 1491 chars, 56 lines
Sections / text-units included: 21

── Section labels (headers first, then body) ──
  [table_cell table[0].row[0].cell[0]]
  [table_cell table[0].row[0].cell[3]]
  [table_cell table[0].row[1].cell[0]]
  [table_cell table[0].row[1].cell[1]]
  [table_cell table[0].row[1].cell[2]]
  [table_cell table[0].row[1].cell[3]]
  [table_cell table[0].row[2].cell[0]]
  [table_cell table[0].row[2].cell[1]]
  [table_cell table[0].row[2].cell[2]]
  [table_cell table[0].row[2].cell[3]]
  [body_paragraph paragraph[5]]
  [body_paragraph paragraph[7]]
  [body_paragraph paragraph[9]]
  [body_paragraph paragraph[11]]
  [table_cell table[1].row[0].cell[0]]
  [РеквизитыДолжностьРуководителяНаименование]
  [table_cell table[1].row[0].cell[1]]
  [РеквизитыРуководительФИО]
  [body_paragraph paragraph[26]]
  [body_paragraph paragraph[27]]
  [body_paragraph paragraph[31]]

── Full text preview (first 1 500 chars) ──
[table_cell table[0].row[0].cell[0]]
БҰЙРЫҚ

[table_cell t

In [8]:
# -- STEP 6: BUILD OPENAI PAYLOAD -----------------------------------------------
from word_constructor.ai_correction.extraction import extract_placeholder_occurrences
from word_constructor.ai_correction.openai_client import _build_payload, SYSTEM_PROMPT
from word_constructor.ai_correction.rules import load_rules_config

rules = load_rules_config()
raw_occurrences = extract_placeholder_occurrences(doc, PARAMS)

payload = _build_payload(
    full_text=full_doc_text,
    occurrences=raw_occurrences,
    rules=rules,
    prompt_ai=PROMPT_AI,
)

print("Model      :", payload["model"])
print("Occurrences sent:", len(raw_occurrences))
print()
print("-- System prompt --")
print(SYSTEM_PROMPT)
print()
print("-- User message (first 3000 chars) --")
print(payload["messages"][1]["content"][:3000])


Model      : gpt-4o-mini
Occurrences sent: 7

-- System prompt --
Ты — редактор официальных деловых документов (приказы, договоры, заявления, акты и др.).

Получаешь:
1. Полный текст документа — колонтитулы, основной текст, таблицы, блок подписи.
2. Список вхождений плейсхолдеров с исходными значениями и окружающим контекстом.
3. Правила коррекции из конфига системы.

Используй полный текст документа, чтобы понять контекст каждого плейсхолдера и вернуть правильную форму значения.

ОБЯЗАТЕЛЬНЫЕ ПРАВИЛА:

ПАДЕЖИ — склоняй по контексту предложения:
  «принять [ФИО]» → вин.п.: «Иванова Ивана Ивановича»
  «от [ФИО]», «заявление [ФИО]» → род.п.: «Иванова Ивана Ивановича»
  «предоставить [ФИО]» → дат.п.: «Иванову Ивану Ивановичу»
  «является [ФИО]» → тв.п.
  Должности в середине предложения тоже склоняй: «принять на должность кассира-повара».

АББРЕВИАТУРЫ — всегда прописными буквами:
  «hr», «Hr», «HR» в должностях → «HR»; аналогично IT, PR, CEO, CFO, CTO и другие бизнес-сокращения.

ДОЛЖНОС

In [9]:
# -- STEP 7: CALL OPENAI --------------------------------------------------------
import time, os
from word_constructor.ai_correction.openai_client import request_ai_corrections
from word_constructor.ai_correction.rules import load_rules_config

call_log = {}
rules = load_rules_config()

print(f"OPENAI_API_KEY present : {bool(os.environ.get('OPENAI_API_KEY','').strip())}")
print(f"Model : {os.environ.get('OPENAI_PLACEHOLDER_MODEL', 'gpt-4o-mini')}")
print()

t0 = time.perf_counter()
ai_corrections = request_ai_corrections(
    full_text=full_doc_text,
    occurrences=raw_occurrences,
    rules=rules,
    prompt_ai=PROMPT_AI,
    log_key="notebook-test",
    call_log=call_log,
    timeout_seconds=30.0,
)
elapsed = time.perf_counter() - t0

print(f"Completed in {elapsed:.2f}s  |  {len(ai_corrections)} corrections")
print()
if "error" in call_log:
    print(f"Error: {call_log['error']}")
for (ph, idx), corrected in ai_corrections.items():
    print(f"  [{ph}]#{idx}  ->  {corrected!r}")


OPENAI_API_KEY present : True
Model : gpt-4o-mini



Completed in 19.83s  |  7 corrections

  [СсылкаСотрудникФизическоеЛицоФИО]#0  ->  'Садыка Ермека Жәнібекұлы'
  [РеквизитыСотрудникДолжностьНаименование]#0  ->  'кассира-повара'
  [РеквизитыСотрудникПодразделениеНаименование]#0  ->  'компас воад'
  [СсылкаДатаПриема]#0  ->  '15 декабря 2025 года'
  [СсылкаСотрудникФизическоеЛицоФИО]#1  ->  'Садыка Ермека Жәнібекұлы'
  [РеквизитыДолжностьРуководителяНаименование]#0  ->  'HR бизнес-партнер'
  [РеквизитыРуководительФИО]#0  ->  'Есжановой Зариной Серикалиевной'


In [10]:
# ── STEP 8: INSPECT RAW OPENAI RESPONSE ───────────────────────────────────
if call_log.get("response"):
    resp = call_log["response"]
    print(f"HTTP status : {resp.get('status')}")
    print(f"Bytes       : {resp.get('bytes')}")
    print()
    print("── Response body (JSON) ──")
    print(json.dumps(resp.get("body", {}), ensure_ascii=False, indent=2)[:3000])
else:
    print("No response recorded in call_log (API key missing or error before request).")

HTTP status : 200
Bytes       : None

── Response body (JSON) ──
{
  "id": "chatcmpl-Dtrfx9WgUqpjsJx42qvcbQggpAaKA",
  "object": "chat.completion",
  "created": 1782206741,
  "model": "gpt-4o-mini-2024-07-18",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "{\"occurrences\":[{\"placeholder\":\"СсылкаСотрудникФизическоеЛицоФИО\",\"occurrence_index\":0,\"corrected_value\":\"Садыка Ермека Жәнібекұлы\",\"changed\":true},{\"placeholder\":\"РеквизитыСотрудникДолжностьНаименование\",\"occurrence_index\":0,\"corrected_value\":\"кассира-повара\",\"changed\":false},{\"placeholder\":\"РеквизитыСотрудникПодразделениеНаименование\",\"occurrence_index\":0,\"corrected_value\":\"компас воад\",\"changed\":false},{\"placeholder\":\"СсылкаДатаПриема\",\"occurrence_index\":0,\"corrected_value\":\"15 декабря 2025 года\",\"changed\":true},{\"placeholder\":\"СсылкаСотрудникФизическоеЛицоФИО\",\"occurrence_index\":1,\"corrected_value\":\"Садыка Ермека

In [11]:
# ── STEP 9: DIFF — ORIGINAL vs AI CORRECTED ───────────────────────────────
# Map correction_id → corrected value, then join with occurrences for display.

id_to_occ = {o["id"]: o for o in raw_occurrences}

print(f"{'PLACEHOLDER':<30} {'OCC':>4}  {'ORIGINAL VALUE':<35} {'AI CORRECTED VALUE':<35} CHANGED")
print("-" * 120)

for occ in raw_occurrences:
    occ_id = occ["id"]
    original = occ["original_value"]
    if occ.get("ai_excluded"):
        corrected = original
        label = "⛔ excluded"
    elif occ.get("fixed_form"):
        corrected = original
        label = "🔒 fixed_form"
    else:
        corrected = ai_corrections.get(occ_id, original)
        label = "✅ changed" if corrected != original else "  same"
    print(f"  [{occ['placeholder']}#{occ['occurrence_index']}]")
    print(f"    Original  : {original!r}")
    print(f"    Corrected : {corrected!r}  {label}")
    print()

PLACEHOLDER                     OCC  ORIGINAL VALUE                      AI CORRECTED VALUE                  CHANGED
------------------------------------------------------------------------------------------------------------------------
  [СсылкаСотрудникФизическоеЛицоФИО#0]
    Original  : 'Садық Ермек Жәнібекұлы'
    Corrected : 'Садық Ермек Жәнібекұлы'    same

  [РеквизитыСотрудникДолжностьНаименование#0]
    Original  : 'кассир-повар'
    Corrected : 'кассир-повар'    same

  [РеквизитыСотрудникПодразделениеНаименование#0]
    Original  : 'компас воад'
    Corrected : 'компас воад'  🔒 fixed_form

  [СсылкаДатаПриема#0]
    Original  : '15.12.2025'
    Corrected : '15.12.2025'    same

  [СсылкаСотрудникФизическоеЛицоФИО#1]
    Original  : 'Садық Ермек Жәнібекұлы'
    Corrected : 'Садық Ермек Жәнібекұлы'    same

  [РеквизитыДолжностьРуководителяНаименование#0]
    Original  : 'hr бизнес-партнер'
    Corrected : 'hr бизнес-партнер'    same

  [РеквизитыРуководительФИО#0]
    Origi

In [12]:
# -- STEP 10: FULL PIPELINE (simplified — AI does everything) -------------------
from word_constructor.ai_correction.pipeline import correct_slot_values

pipeline_log = {}
result = correct_slot_values(
    doc=doc,
    slot_values=PARAMS,
    prompt_ai=PROMPT_AI,
    log_key="notebook-pipeline",
    call_log=pipeline_log,
    timeout_seconds=30.0,
)

print(f"Pipeline done  |  {len(result.occurrence_values)} values corrected")
print()
print(f"  {'PLACEHOLDER':<38} {'ORIGINAL':<32} {'FINAL':<32}")
print("  " + "-" * 105)
for key, original in PARAMS.items():
    finals = {n: v for (k, n), v in result.occurrence_values.items() if k == key}
    if finals:
        for occ_num, final_val in sorted(finals.items()):
            marker = "CHANGED" if final_val != original else "      -"
            print(f"  {marker}  {key:<36} {repr(original):<32} {repr(final_val):<32}")
    else:
        print(f"  (no change)  {key:<36} {repr(original)}")


Pipeline done  |  7 values corrected

  PLACEHOLDER                            ORIGINAL                         FINAL                           
  ---------------------------------------------------------------------------------------------------------
  (no change)  Ссылка                               'Прием на работу 0000-000233 от 15.12.2025'
  (no change)  СсылкаДолжностьРуководителяНаименование 'hr бизнес-партнер'
  CHANGED  СсылкаДатаПриема                     '15.12.2025'                     '15 декабря 2025 года'          
        -  РеквизитыСотрудникПодразделениеНаименование 'компас воад'                    'компас воад'                   
  CHANGED  СсылкаСотрудникФизическоеЛицоФИО     'Садық Ермек Жәнібекұлы'         'Садыка Ермека Жәнібекұлы'      
  CHANGED  СсылкаСотрудникФизическоеЛицоФИО     'Садық Ермек Жәнібекұлы'         'Садыка Ермека Жәнібекұлы'      
  (no change)  РеквизитыОтветственныйФизическоеЛицоФИО 'Сатымбаева Салтанат Мусаевна'
  CHANGED  РеквизитыРуковод

In [13]:
# -- STEP 11: COMPREHENSIVE DEMO (mock document, all via AI) ----------------
import io
from docx import Document as _Doc
from word_constructor.ai_correction.pipeline import correct_slot_values

_buf = io.BytesIO()
_d = _Doc()
_d.add_paragraph("ДОГОВОР No [НомерДоговора] от [ДатаДоговора] года")
_d.add_paragraph(
    "Договор заключен сроком на [КолДней] [КолДнейПрописью] "
    "календарных дней, с [ДатаНачала] года по [ДатаОкончания] года."
)
_d.add_paragraph("В течение [СрокДней] рабочих дней предоставить документы.")
_d.add_paragraph(
    "Принять [ФИОСотрудника] на должность [ДолжностьСотрудника] "
    "[ПодразделениеСотрудника]."
)
_d.add_paragraph("Заявление от [ФИОСотрудника2].")
_tbl = _d.add_table(rows=1, cols=2)
_tbl.rows[0].cells[0].text = "[ДолжностьРуководителя]"
_tbl.rows[0].cells[1].text = "[ФИОРуководителя]"
_d.save(_buf)
_buf.seek(0)
_demo_doc = _Doc(_buf)

demo_params = {
    "НомерДоговора":           "ДОГ-2025-001",
    "ДатаДоговора":            "15.12.2025",
    "КолДней":                 "30",
    "КолДнейПрописью":         "30",
    "ДатаНачала":              "01.01.2026",
    "ДатаОкончания":           "31.12.2026",
    "СрокДней":                "10",
    "ФИОСотрудника":           "Есжанова Зарина Серикалиевна",
    "ДолжностьСотрудника":     "кассир-повар",
    "ПодразделениеСотрудника": "Отдел кадров",
    "ФИОСотрудника2":          "Есжанова Зарина Серикалиевна",
    "ДолжностьРуководителя":   "hr бизнес-партнер",
    "ФИОРуководителя":         "Есжанова Зарина Серикалиевна",
}

print("Running simplified AI pipeline on comprehensive demo...")
print()
demo_result = correct_slot_values(
    doc=_demo_doc,
    slot_values=demo_params,
    prompt_ai="",
    log_key="notebook-demo",
    timeout_seconds=30.0,
)

print(f"  {'PLACEHOLDER':<30} {'ORIGINAL':<35} {'AI RESULT':<35}")
print("  " + "-" * 102)
for key, original in demo_params.items():
    finals = {n: v for (k, n), v in demo_result.occurrence_values.items() if k == key}
    if finals:
        for occ_num, final_val in sorted(finals.items()):
            marker = "CHANGED" if final_val != original else "      -"
            print(f"  {marker}  {key:<28} {repr(original):<35} {repr(final_val):<35}")
    else:
        print(f"  (no change)  {key:<28} {repr(original)}")


Running simplified AI pipeline on comprehensive demo...



  PLACEHOLDER                    ORIGINAL                            AI RESULT                          
  ------------------------------------------------------------------------------------------------------
        -  НомерДоговора                'ДОГ-2025-001'                      'ДОГ-2025-001'                     
  CHANGED  ДатаДоговора                 '15.12.2025'                        '15 декабря 2025 года'             
        -  КолДней                      '30'                                '30'                               
  CHANGED  КолДнейПрописью              '30'                                'тридцать'                         
  CHANGED  ДатаНачала                   '01.01.2026'                        '1 января 2026 года'               
  CHANGED  ДатаОкончания                '31.12.2026'                        '31 декабря 2026 года'             
        -  СрокДней                     '10'                                '10'                               
      

In [14]:
# -- STEP 12: LOG ANALYSIS — last 50 runs -----------------------------------
from word_constructor.ai_correction import log_store

entries = log_store.read_all()
print(f"Total stored log entries: {len(entries)}")

if not entries:
    print("No log entries yet — run STEP 10 or STEP 11 first.")
else:
    try:
        import pandas as pd
        rows = []
        for e in entries:
            timing = e.get("timing_ms") or {}
            tokens = e.get("tokens") or {}
            rows.append({
                "ts":      (e.get("ts") or "")[:19],
                "log_key": (e.get("log_key") or "")[:28],
                "status":  e.get("status", ""),
                "slots":   e.get("slot_count", 0),
                "occ":     e.get("occurrence_count", 0),
                "changed": e.get("changed_count", 0),
                "ai_ms":   (e.get("timing_ms") or {}).get("ai_call", ""),
                "tokens":  (e.get("tokens") or {}).get("total_tokens", ""),
                "model":   (e.get("model") or "")[-14:],
            })
        df = pd.DataFrame(rows)
        print("Last 10 runs:")
        display(df.tail(10))
    except ImportError:
        for e in entries[-5:]:
            timing = e.get("timing_ms") or {}
            status = e.get("status", "")
            changed = e.get("changed_count", 0)
            occ = e.get("occurrence_count", 0)
            ai_ms = timing.get("ai_call", "?")
            ts_ = (e.get("ts") or "")[:19]
            lk_ = (e.get("log_key") or "")[:24]
            print(f"  {ts_}  key={lk_}  status={status}  changed={changed}/{occ}  ai_ms={ai_ms}ms")

    latest = entries[-1]
    ts = (latest.get("ts") or "")[:19]
    lk = latest.get("log_key") or ""
    print("")
    print(f"-- Latest: {ts} | key={lk} | status={latest.get('status')} --")
    for corr in latest.get("corrections", []):
        marker = "CHANGED" if corr.get("changed") else "      -"
        ph    = corr.get("placeholder", "")
        idx   = corr.get("occurrence_index", 0)
        orig  = corr.get("original", "")
        final = corr.get("final", "")
        if corr.get("changed"):
            change_str = repr(orig) + " --> " + repr(final)
        else:
            change_str = repr(orig)
        print(f"  {marker}  [{ph}]#{idx}  {change_str}")
    print("")
    timing = latest.get("timing_ms") or {}
    tokens = latest.get("tokens") or {}
    ai_ms = timing.get("ai_call", "?")
    tot_ms = timing.get("total", "?")
    pt = tokens.get("prompt_tokens", "?")
    ct = tokens.get("completion_tokens", "?")
    tt = tokens.get("total_tokens", "?")
    print(f"  Timing: ai={ai_ms}ms  total={tot_ms}ms")
    print(f"  Tokens: prompt={pt}  completion={ct}  total={tt}")
    print(f"  Log file: {log_store._log_path()}")


Total stored log entries: 10


Last 10 runs:


,ts,log_key,status,slots,occ,changed,ai_ms,tokens,model
0,2026-06-23T07:42:49,test-run-001,ok,12,7,5,6503,,gpt-4o-mini
1,2026-06-23T08:10:31,test-fix,ok,12,7,5,3833,,gpt-4o-mini
2,2026-06-23T08:16:02,test-title-fix,ok,12,7,6,5995,,gpt-4o-mini
3,2026-06-23T09:14:05,simplified-test,ok,12,7,3,6967,,gpt-4o-mini
4,2026-06-23T09:23:28,notebook-pipeline,ok,9,7,6,6328,3167,gpt-4o-mini
5,2026-06-23T09:23:38,notebook-demo,ok,13,13,7,10734,,gpt-4o-mini
6,2026-06-23T09:24:32,notebook-pipeline,ok,9,7,6,8045,3167,gpt-4o-mini
7,2026-06-23T09:24:47,notebook-demo,ok,13,13,7,14917,,gpt-4o-mini
8,2026-06-23T09:26:08,notebook-pipeline,ok,9,7,6,6173,3167,gpt-4o-mini
9,2026-06-23T09:26:26,notebook-demo,ok,13,13,7,17169,,gpt-4o-mini



-- Latest: 2026-06-23T09:26:26 | key=notebook-demo | status=ok --
        -  [НомерДоговора]#0  'ДОГ-2025-001'
  CHANGED  [ДатаДоговора]#0  '15.12.2025' --> '15 декабря 2025 года'
        -  [КолДней]#0  '30'
  CHANGED  [КолДнейПрописью]#0  '30' --> 'тридцать'
  CHANGED  [ДатаНачала]#0  '01.01.2026' --> '1 января 2026 года'
  CHANGED  [ДатаОкончания]#0  '31.12.2026' --> '31 декабря 2026 года'
        -  [СрокДней]#0  '10'
        -  [ФИОСотрудника]#0  'Есжанова Зарина Серикалиевна'
  CHANGED  [ДолжностьСотрудника]#0  'кассир-повар' --> 'Кассир-повар'
  CHANGED  [ПодразделениеСотрудника]#0  'Отдел кадров' --> ''
        -  [ФИОСотрудника2]#0  'Есжанова Зарина Серикалиевна'
  CHANGED  [ДолжностьРуководителя]#0  'hr бизнес-партнер' --> 'HR бизнес-партнер'
        -  [ФИОРуководителя]#0  'Есжанова Зарина Серикалиевна'

  Timing: ai=17169ms  total=17171ms
  Tokens: prompt=?  completion=?  total=?
  Log file: C:\Users\User\Desktop\word-portal\tmp_logs\corrections.jsonl
